In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
%store -r restrict_df

df = restrict_df

df["CloseDate"] = pd.to_datetime(
    df["CloseDate"], errors="coerce"
)

df["ClosePrice"] = pd.to_numeric(
    df["ClosePrice"], errors="coerce"
)

df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

# A supervised training row needs a valid target and date.
df = df.dropna(subset=["ClosePrice", "CloseDate"])

# Exclude impossible/non-positive sale prices.
df = df[df["ClosePrice"] > 0].copy()

print("Filtered shape:", df.shape)
print(df["CloseDate"].min(), df["CloseDate"].max())
print(df["ClosePrice"].describe())

Filtered shape: (65344, 80)
2024-02-01 00:00:00 2024-12-31 00:00:00
count    6.534400e+04
mean     1.221108e+06
std      1.142921e+06
min      1.150000e+00
25%      6.209900e+05
50%      8.888440e+05
75%      1.410001e+06
max      1.965000e+07
Name: ClosePrice, dtype: float64


In [3]:
missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing["missing_pct"] = (
    missing["missing_count"] / len(df) * 100
)

missing = missing.sort_values(
    "missing_pct", ascending=False
)

display(missing.head(30))

print("Exact duplicate rows:", df.duplicated().sum())

if "ListingKey" in df.columns:
    print("Unique listings:", df["ListingKey"].nunique())

,missing_count,missing_pct
TaxYear,65344,100.000000
ElementarySchoolDistrict,65344,100.000000
BusinessType,65344,100.000000
CoveredSpaces,65344,100.000000
MiddleOrJuniorSchoolDistrict,65344,100.000000
TaxAnnualAmount,65344,100.000000
AboveGradeFinishedArea,65344,100.000000
FireplacesTotal,65344,100.000000
WaterfrontYN,65327,99.973984
BelowGradeFinishedArea,64969,99.426114


Exact duplicate rows: 22
Unique listings: 65282


In [8]:
numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt"
]

categorical_features = [
    "PostalCode"
]

# Keep only columns present in your actual dataset.
numeric_features = [
    c for c in numeric_features if c in df.columns
]

categorical_features = [
    c for c in categorical_features if c in df.columns
]

feature_cols = numeric_features + categorical_features

X = df[feature_cols].copy()
y = df["ClosePrice"].copy()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt']
Categorical features: ['PostalCode']


In [9]:
# Use the latest calendar month in the dataset as the test month.
test_start = df["CloseDate"].max().to_period("M").start_time
test_end = test_start + pd.DateOffset(months=1)

# Initial experiment: use the preceding 5 months for training.
TRAIN_MONTHS = 5
train_start = test_start - pd.DateOffset(
    months=TRAIN_MONTHS
)

train_mask = (
    (df["CloseDate"] >= train_start) &
    (df["CloseDate"] < test_start)
)

test_mask = (
    (df["CloseDate"] >= test_start) &
    (df["CloseDate"] < test_end)
)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training period:", train_start, "to", test_start)
print("Testing period:", test_start, "to", test_end)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training period: 2024-07-01 00:00:00 to 2024-12-01 00:00:00
Testing period: 2024-12-01 00:00:00 to 2025-01-01 00:00:00
Training rows: 45565
Testing rows: 10417


In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (45565, 1516)
Processed testing shape: (10417, 1516)


In [11]:
# Create a CSV containing numerically encoded features
feature_names = preprocessor.get_feature_names_out()

X_processed = preprocessor.transform(X)

# Convert sparse output to a dense array if necessary
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

processed_df = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=X.index
)

# Combine the original target/date columns with encoded features
cleaned = pd.concat(
    [
        df[["CloseDate", "ClosePrice"]],
        processed_df
    ],
    axis=1
)

cleaned.to_csv("cleaned_properties.csv", index=False)

print("Saved cleaned_properties.csv")

Saved cleaned_properties.csv
